In [2]:
import numpy as np
import CoRT_builder
import utils
from sklearn.linear_model import Lasso
from scipy.stats import norm

In [9]:
n_target = 120
n_source = 20
p = 100
K = 5
Ka = 2
h = 15
alpha = 0.05
T = 3
s_len = 10
s_vector = [0.7] * s_len
N = n_target / 2 + Ka * n_source
NI = n_target / 2 + n_source
lamda_k_source = 2 * np.sqrt(np.log(p)/ N)
lamda_1_source = 2 * np.sqrt(np.log(p)/ NI) 
lamda_not_source = 2 * np.sqrt(np.log(p) / (n_target / 2)) 
iteration = 1000

CoRT_model = CoRT_builder.CoRT(lamda_not_source)
para_results_storage = []

for i in range(0, iteration):
    target_data, source_data = CoRT_model.gen_data(n_target, n_source, p, K, Ka, h, s_vector, s_len, "AR")
    # splitting data
    if i % 50 == 0:
        print(f"Iteration {i}")
    X_target = target_data["X"]
    y_target = target_data["y"]
    folds = utils.split_target(2, X_target, y_target, n_target)
    target_data_train = folds[0]
    target_data_test = folds[1]
    n_target_new = int(n_target / 2)

    similar_source_index = CoRT_model.find_similar_source(n_target_new, K, target_data_train, source_data, lamda_not_source, lamda_1_source, T=T, verbose=False)
    X_combined, y_combined = CoRT_model.prepare_CoRT_data(similar_source_index, source_data, target_data_train)
    model = Lasso(alpha=lamda_k_source, fit_intercept=False, tol=1e-10, max_iter=100000)
    model.fit(X_combined, y_combined.ravel())
    beta_hat_target = model.coef_[-p:]

    M_obs = np.array([i for i, b in enumerate(beta_hat_target) if b != 0])
    if len(M_obs) == 0:
        print(f"Iteration {iter}: Lasso selected no features. Skipping.")
        continue

    j = np.random.choice(len(M_obs))
    selected_feature_index = M_obs[j]

    X_target = target_data_test["X"]
    y_target = target_data_test["y"]
    X_active, X_inactive = utils.get_active_X(beta_hat_target, X_target)
    etaj, etajTy = utils.construct_test_statistic(y_target, j, X_active)
    Sigma = np.eye(n_target_new)
    tn_sigma = (np.sqrt(etaj.T @ Sigma @ etaj)).item()

    p_value = 2 * (1 - norm.cdf(abs(etajTy), loc = 0, scale = tn_sigma))

    is_signal = (selected_feature_index < s_len) 
    result_dict = {
        "p_value": p_value,
        "is_signal": is_signal,
        "feature_idx": selected_feature_index
    }
    print(f"is_signal : {result_dict['is_signal']}, p_values[{i}]: {result_dict['p_value']}")
    para_results_storage.append(result_dict)

is_signal_cases = [r for r in para_results_storage if r['is_signal']]
not_signal_cases = [r for r in para_results_storage if not r['is_signal']]

false_positives = sum(1 for c in not_signal_cases if c['p_value'] <= alpha)
fpr = false_positives / len(not_signal_cases)
true_positives = sum(1 for r in is_signal_cases if r['p_value'] <= alpha)
tpr = true_positives / len(is_signal_cases)

print(f"FPR: {fpr}")
print(f"TPR: {tpr}")

Iteration 0
is_signal : True, p_values[0]: 0.0006058605203305145
is_signal : True, p_values[1]: 0.0017826921359462844
is_signal : True, p_values[2]: 2.7284257964055314e-08
is_signal : True, p_values[3]: 0.005333343101856514
is_signal : True, p_values[4]: 1.308721640036481e-05
is_signal : True, p_values[5]: 0.006411966112277456
is_signal : True, p_values[6]: 2.359940021179341e-09
is_signal : True, p_values[7]: 0.003246115484470735
is_signal : True, p_values[8]: 0.00048633883049831184
is_signal : True, p_values[9]: 0.00027297210721477505
is_signal : True, p_values[10]: 0.000190591538661522
is_signal : True, p_values[11]: 0.048990329452231585
is_signal : True, p_values[12]: 0.0007307731863244182
is_signal : True, p_values[13]: 0.19506151099921376
is_signal : True, p_values[14]: 3.834062778373948e-08
is_signal : True, p_values[15]: 0.018859009060939735
is_signal : False, p_values[16]: 0.17149889965755793
is_signal : True, p_values[17]: 5.0654189544863115e-05
is_signal : True, p_values[18]:

In [5]:
print(f"FPR: {fpr}")
print(f"TPR: {tpr}")

FPR: 0.4
TPR: 0.7413793103448276
